<a href="https://colab.research.google.com/github/JdelaM13/gestion-ordenes/blob/main/API_4_Base_de_Datos_y_Big_Data_GORDILLO_Juan_Manuel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **API 4 - BASE DE DATOS Y BIG DATA**

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

## **Creación de Tablas con Spark (DDL)**

* **Equivalente a la API 3.**

### Inicializar la Sesión de Spark

In [16]:
"""
Proyecto: Data Warehouse para Farispley - Actividad 4
Script: Creación de estructuras de Data Vault como DataFrames de Spark.
Equivalente al DDL (CREATE TABLE) en SQL de la Actividad 3.
"""


###################################
# 1. Inicializar la Sesión de Spark
###################################

# Esto es lo primero que se hace en cualquier aplicación de Spark.
# Le da a nuestro programa el punto de entrada para comunicarse con el clúster de Spark.

spark = SparkSession.builder \
    .appName("DataVaultDML_Farispley") \
    .master("local[*]") \
    .getOrCreate()

# .appName(): Asigna un nombre a nuestra aplicación. Es útil para identificarla en la interfaz de usuario de Spark.

# .master(): Especifica a qué clúster de Spark nos conectamos.
# "local[*]": Significa que estamos ejecutando Spark en modo local en nuestra propia máquina,
# utilizando todos los núcleos de CPU disponibles para simular un entorno paralelo. Ideal para desarrollo.

# .getOrCreate(): Construye y devuelve el objeto SparkSession.
# Si ya existe una sesión con la misma configuración, la reutiliza; si no, crea una nueva.

### Definir los Esquemas (Schemas) para cada tabla del Data Vault

In [17]:
###################################
# 2. Definir los Esquemas (Schemas) para cada tabla del Data Vault
###################################
# Un esquema define los nombres de las columnas y sus tipos de datos.
# Es una buena práctica definir explícitamente los esquemas para asegurar la calidad de los datos.

###################################
# Esquemas para los HUBS
###################################
hub_articulo_schema = StructType([
    StructField("HK_ARTICULO", StringType(), False),
    StructField("BK_IDARTICULO", IntegerType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("RECORD_SOURCE", StringType(), False)
])

hub_tipo_articulo_schema = StructType([
    StructField("HK_TIPO_ARTICULO", StringType(), False),
    StructField("BK_MNA", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("RECORD_SOURCE", StringType(), False)
])

hub_stock_fisico_schema = StructType([
    StructField("HK_STOCK_FISICO", StringType(), False),
    StructField("BK_IDSTOCK", IntegerType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("RECORD_SOURCE", StringType(), False)
])

###################################
# Esquemas para los LINKS
###################################
lnk_articulo_tipo_articulo_schema = StructType([
    StructField("LK_ARTICULO_TIPO_PK", StringType(), False),
    StructField("HK_ARTICULO", StringType(), False),
    StructField("HK_TIPO_ARTICULO", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("RECORD_SOURCE", StringType(), False)
])

lnk_articulo_stock_fisico_schema = StructType([
    StructField("LK_ARTICULO_STOCK_PK", StringType(), False),
    StructField("HK_ARTICULO", StringType(), False),
    StructField("HK_STOCK_FISICO", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("RECORD_SOURCE", StringType(), False)
])

###################################
# Esquemas para los SATELLITES
###################################
sat_articulo_detalles_schema = StructType([
    StructField("HK_ARTICULO", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("HASH_DIFF", StringType(), False),
    StructField("SectSecc", IntegerType(), True),
    StructField("idnumero_desc", IntegerType(), True),
    StructField("CodigoSAP_desc", IntegerType(), True),
    StructField("Terminal", StringType(), True),
    StructField("Usuario", StringType(), True),
    StructField("TS_ORIGEN", TimestampType(), True),
    StructField("RECORD_SOURCE", StringType(), False)
])

sat_tipo_articulo_detalles_schema = StructType([
    StructField("HK_TIPO_ARTICULO", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("HASH_DIFF", StringType(), False),
    StructField("Descrip", StringType(), True),
    StructField("TipoCotizacion", StringType(), True),
    StructField("Terminal", StringType(), True),
    StructField("Usuario", StringType(), True),
    StructField("TS_ORIGEN", TimestampType(), True),
    StructField("RECORD_SOURCE", StringType(), False)
])

sat_stock_fisico_detalles_schema = StructType([
    StructField("HK_STOCK_FISICO", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("HASH_DIFF", StringType(), False),
    StructField("Descrip", StringType(), True),
    StructField("Apto", StringType(), True),
    StructField("Terminal", StringType(), True),
    StructField("Usuario", StringType(), True),
    StructField("TS_ORIGEN", TimestampType(), True),
    StructField("RECORD_SOURCE", StringType(), False)
])

sat_lnk_articulo_stock_estado_schema = StructType([
    StructField("LK_ARTICULO_STOCK_PK", StringType(), False),
    StructField("LOAD_DATE", TimestampType(), False),
    StructField("HASH_DIFF", StringType(), False),
    StructField("Scanning", IntegerType(), True),
    StructField("UxB", IntegerType(), True),
    StructField("CantStock", IntegerType(), True),
    StructField("Unidades", IntegerType(), True),
    StructField("Terminal", StringType(), True),
    StructField("Usuario", StringType(), True),
    StructField("TS_ORIGEN", TimestampType(), True),
    StructField("RECORD_SOURCE", StringType(), False)
])

### Crear DataFrames vacíos con los esquemas definidos

In [18]:
###################################
# 3. Crear DataFrames vacíos con los esquemas definidos
###################################
# Esto nos da la estructura lista para ser poblada con datos.
hub_articulo_df = spark.createDataFrame([], schema=hub_articulo_schema)
hub_tipo_articulo_df = spark.createDataFrame([], schema=hub_tipo_articulo_schema)
hub_stock_fisico_df = spark.createDataFrame([], schema=hub_stock_fisico_schema)

lnk_articulo_tipo_articulo_df = spark.createDataFrame([], schema=lnk_articulo_tipo_articulo_schema)
lnk_articulo_stock_fisico_df = spark.createDataFrame([], schema=lnk_articulo_stock_fisico_schema)

sat_articulo_detalles_df = spark.createDataFrame([], schema=sat_articulo_detalles_schema)
sat_tipo_articulo_detalles_df = spark.createDataFrame([], schema=sat_tipo_articulo_detalles_schema)
sat_stock_fisico_detalles_df = spark.createDataFrame([], schema=sat_stock_fisico_detalles_schema)
sat_lnk_articulo_stock_estado_df = spark.createDataFrame([], schema=sat_lnk_articulo_stock_estado_schema)

### Verificamos y detenemos la sesión de Spark

In [19]:
###################################
# 4. Verificación (Opcional)
###################################
# Podemos imprimir los esquemas para verificar que se crearon correctamente.
print("Esquema de HUB_ARTICULO:")
hub_articulo_df.printSchema()

print("\nEsquema de SAT_LNK_ARTICULO_STOCK_ESTADO:")
sat_lnk_articulo_stock_estado_df.printSchema()
###################################
# 5. Detener la Sesión de Spark
###################################
# Es una buena práctica liberar los recursos al final del script.
# spark.stop() # Removed to keep the session active for subsequent cells

Esquema de HUB_ARTICULO:
root
 |-- HK_ARTICULO: string (nullable = false)
 |-- BK_IDARTICULO: integer (nullable = false)
 |-- LOAD_DATE: timestamp (nullable = false)
 |-- RECORD_SOURCE: string (nullable = false)


Esquema de SAT_LNK_ARTICULO_STOCK_ESTADO:
root
 |-- LK_ARTICULO_STOCK_PK: string (nullable = false)
 |-- LOAD_DATE: timestamp (nullable = false)
 |-- HASH_DIFF: string (nullable = false)
 |-- Scanning: integer (nullable = true)
 |-- UxB: integer (nullable = true)
 |-- CantStock: integer (nullable = true)
 |-- Unidades: integer (nullable = true)
 |-- Terminal: string (nullable = true)
 |-- Usuario: string (nullable = true)
 |-- TS_ORIGEN: timestamp (nullable = true)
 |-- RECORD_SOURCE: string (nullable = false)





---



## **Inserción de Datos Sintéticos con Spark (DML)**

* **Equivalente a la API 3**

### Inicializar la Sesión de Spark

In [20]:
"""
Proyecto: Data Warehouse para Farispley - Actividad 4
Script: Inserción de datos sintéticos en DataFrames de Spark.
Equivalente al DML (INSERT INTO) en SQL.
"""

from pyspark.sql import SparkSession

from datetime import (
    datetime,   # Representa un punto específico en el tiempo (fecha y hora). Se usa para obtener la fecha actual con `datetime.now()`.
    timedelta   # Representa una duración o un intervalo de tiempo. Se usa para sumar o restar días/horas a una fecha (ej: `timedelta(days=10)`).
)

###################################
# 1. Inicializar la Sesión de Spark
###################################
spark = SparkSession.builder \
    .appName("DataVaultDML_Farispley") \
    .master("local[*]") \
    .getOrCreate()

### Definir los datos sintéticos

In [21]:
###################################
# 2. Definir los datos sintéticos
###################################
now = datetime.now()

###################################
# Datos para los HUBS
###################################
hub_articulo_data = [('d8b7390c35f6311e60a37f5960413a21', 75001, now - timedelta(days=10), 'MNADARTICULOS'), ('5e1b2413a521c3a1b327b4e803dc42b6', 83045, now - timedelta(days=10), 'MNADARTICULOS')]
hub_tipo_articulo_data = [('255f053229b3699313d33e1471018331', 'ELEC', now - timedelta(days=10), 'MNAdTipoArticulos'), ('11e3c26b52c1a6a32655359a6137a9a9', 'INFO', now - timedelta(days=10), 'MNAdTipoArticulos')]
hub_stock_fisico_data = [('d3d9446802a44259755d38e6d163e820', 10, now - timedelta(days=10), 'MNLdStockFisico'), ('98f13708210194c475687be6106a3b84', 20, now - timedelta(days=10), 'MNLdStockFisico')]

###################################
# Datos para los SATELLITES de los HUBS
###################################
sat_articulo_detalles_data = [('d8b7390c35f6311e60a37f5960413a21', now - timedelta(days=10), '970591f42778a87313a48436b1d446a7', 1, 9001, 1001, None, None, None, 'MNADARTICULOS'), ('5e1b2413a521c3a1b327b4e803dc42b6', now - timedelta(days=10), '6f48f4306c59b81602485303c62383c2', 2, 9002, 1002, None, None, None, 'MNADARTICULOS')]
sat_tipo_articulo_detalles_data = [('255f053229b3699313d33e1471018331', now - timedelta(days=10), '9341459a96e3868615b3a328e1d55643', 'Electrónica', 'Pesos', None, None, None, 'MNAdTipoArticulos'), ('11e3c26b52c1a6a32655359a6137a9a9', now - timedelta(days=10), '1585721151f7e0964c5750d4f3b255bc', 'Informática', 'Dolar', None, None, None, 'MNAdTipoArticulos')]
sat_stock_fisico_detalles_data = [('d3d9446802a44259755d38e6d163e820', now - timedelta(days=10), '21098a531e0f0d2c8033230c1737e3d8', 'Depósito Central', 'SI', None, None, None, 'MNLdStockFisico'), ('98f13708210194c475687be6106a3b84', now - timedelta(days=10), '837c7c34d9a1506b38a4f215d86242c7', 'Tienda Unicenter', 'SI', None, None, None, 'MNLdStockFisico')]

###################################
# Datos para los LINKS
###################################
lnk_articulo_tipo_articulo_data = [('b83b3814a09337b524c51475c75c873a', 'd8b7390c35f6311e60a37f5960413a21', '255f053229b3699313d33e1471018331', now - timedelta(days=10), 'MNADARTICULOS'), ('3b4685de79b5c2c77d488e36e336c117', '5e1b2413a521c3a1b327b4e803dc42b6', '11e3c26b52c1a6a32655359a6137a9a9', now - timedelta(days=10), 'MNADARTICULOS')]
lnk_articulo_stock_fisico_data = [('64733334e359d99767f41f7e44a19b84', 'd8b7390c35f6311e60a37f5960413a21', 'd3d9446802a44259755d38e6d163e820', now - timedelta(days=5), 'SIP..SIPdStocks'), ('7784013c7a0910996f485f543c332509', 'd8b7390c35f6311e60a37f5960413a21', '98f13708210194c475687be6106a3b84', now - timedelta(days=5), 'SIP..SIPdStocks'), ('8e906c525f0e11894a47814b7e88701e', '5e1b2413a521c3a1b327b4e803dc42b6', '98f13708210194c475687be6106a3b84', now - timedelta(days=5), 'SIP..SIPdStocks')]

###################################
# Datos para el SATELLITE del LINK (El historial de stock)
###################################
sat_lnk_articulo_stock_estado_data = [
    ('64733334e359d99767f41f7e44a19b84', now - timedelta(days=2), '6c81804f5e3e57f12e3f898a35606e98', 123, 1, 160, 160, None, None, None, 'SIP..SIPdStocks'),
    ('64733334e359d99767f41f7e44a19b84', now - timedelta(days=1), 'b53dbd76e4c77c080e7228833b743329', 123, 1, 150, 150, None, None, None, 'SIP..SIPdStocks'),
    ('7784013c7a0910996f485f543c332509', now - timedelta(days=1, hours=4), 'a18836263544f38697621c1103f568a9', 124, 1, 15, 15, None, None, None, 'SIP..SIPdStocks'),
    ('7784013c7a0910996f485f543c332509', now - timedelta(days=1), '20f71c15f913d96144e1302d99d251f9', 124, 1, 12, 12, None, None, None, 'SIP..SIPdStocks'),
    ('8e906c525f0e11894a47814b7e88701e', now - timedelta(days=3), '7b7a13437e58a27d2c3882bbd515cd2c', 125, 1, 30, 30, None, None, None, 'SIP..SIPdStocks'),
    ('8e906c525f0e11894a47814b7e88701e', now - timedelta(days=1), '7ca633917812903806f147041a941566', 125, 1, 25, 25, None, None, None, 'SIP..SIPdStocks')
]

### Crear los DataFrames a partir de los datos y esquemas

In [22]:
###################################
# 3. Crear los DataFrames a partir de los datos y esquemas
###################################

hub_articulo_df = spark.createDataFrame(data=hub_articulo_data, schema=hub_articulo_schema)
hub_tipo_articulo_df = spark.createDataFrame(data=hub_tipo_articulo_data, schema=hub_tipo_articulo_schema)
hub_stock_fisico_df = spark.createDataFrame(data=hub_stock_fisico_data, schema=hub_stock_fisico_schema)

sat_articulo_detalles_df = spark.createDataFrame(data=sat_articulo_detalles_data, schema=sat_articulo_detalles_schema)
sat_tipo_articulo_detalles_df = spark.createDataFrame(data=sat_tipo_articulo_detalles_data, schema=sat_tipo_articulo_detalles_schema)
sat_stock_fisico_detalles_df = spark.createDataFrame(data=sat_stock_fisico_detalles_data, schema=sat_stock_fisico_detalles_schema)

lnk_articulo_tipo_articulo_df = spark.createDataFrame(data=lnk_articulo_tipo_articulo_data, schema=lnk_articulo_tipo_articulo_schema)
lnk_articulo_stock_fisico_df = spark.createDataFrame(data=lnk_articulo_stock_fisico_data, schema=lnk_articulo_stock_fisico_schema)

sat_lnk_articulo_stock_estado_df = spark.createDataFrame(data=sat_lnk_articulo_stock_estado_data, schema=sat_lnk_articulo_stock_estado_schema)


### Verificamos y detenemos la sesión de Spark

In [23]:
###################################
# 4. Verificación (Opcional)
###################################
# Mostramos algunos datos para confirmar que se cargaron bien.
print("Datos de HUB_ARTICULO:")
hub_articulo_df.show()

print("\nDatos de SAT_LNK_ARTICULO_STOCK_ESTADO (historial de stock):")
sat_lnk_articulo_stock_estado_df.show(truncate=False)
###################################
# 5. Detener la Sesión de Spark
###################################

spark.stop()

Datos de HUB_ARTICULO:
+--------------------+-------------+--------------------+-------------+
|         HK_ARTICULO|BK_IDARTICULO|           LOAD_DATE|RECORD_SOURCE|
+--------------------+-------------+--------------------+-------------+
|d8b7390c35f6311e6...|        75001|2026-05-04 07:17:...|MNADARTICULOS|
|5e1b2413a521c3a1b...|        83045|2026-05-04 07:17:...|MNADARTICULOS|
+--------------------+-------------+--------------------+-------------+


Datos de SAT_LNK_ARTICULO_STOCK_ESTADO (historial de stock):
+--------------------------------+--------------------------+--------------------------------+--------+---+---------+--------+--------+-------+---------+---------------+
|LK_ARTICULO_STOCK_PK            |LOAD_DATE                 |HASH_DIFF                       |Scanning|UxB|CantStock|Unidades|Terminal|Usuario|TS_ORIGEN|RECORD_SOURCE  |
+--------------------------------+--------------------------+--------------------------------+--------+---+---------+--------+--------+-------

# **RESOLUCIÓN**

### Inicializar la Sesión de Spark

In [24]:
# -*- coding: utf-8 -*-
"""
Proyecto: Data Warehouse para Farispley - Actividad 4
Script: Resolución de Datamarts utilizando Spark SQL.
"""

from pyspark.sql import SparkSession

from pyspark.sql.window import Window
# Window: Se usa para definir la "ventana" o el grupo de filas sobre el cual operarán las funciones de ventana (ej: define el PARTITION BY y ORDER BY).

from pyspark.sql.functions import (
    col,          # Permite referirse a una columna por su nombre (ej: col("nombre_columna")). Esencial para la API de DataFrames.
    row_number,   # La función de ventana que asigna un número secuencial (1, 2, 3...) a las filas dentro de una partición.
    current_date, # Devuelve la fecha actual del sistema.
    date_sub,     # Resta un número específico de días a una columna de tipo fecha.
    lit           # Crea una columna con un valor literal o constante (ej: lit("Hola Mundo")).
)


###################################
# --- PASO 1: INICIALIZACIÓN Y CARGA DE DATOS ---
###################################
# En un escenario real, los datos se leerían de un Data Lake (ej: Parquet, Delta Lake).
# Para este ejercicio, reutilizamos el código de inserción para tener los DataFrames disponibles.

spark = SparkSession.builder \
    .appName("DatamartSpark_Farispley") \
    .master("local[*]") \
    .getOrCreate()

### REGISTRAR DATAFRAMES COMO VISTAS TEMPORALES

In [25]:
###################################
# --- PASO 2: REGISTRAR DATAFRAMES COMO VISTAS TEMPORALES ---
###################################

#Se aplica el esquema correspondiente a cada DataFrame al momento de su creación.
hub_articulo_df = spark.createDataFrame(data=hub_articulo_data, schema=hub_articulo_schema)
hub_tipo_articulo_df = spark.createDataFrame(data=hub_tipo_articulo_data, schema=hub_tipo_articulo_schema)
hub_stock_fisico_df = spark.createDataFrame(data=hub_stock_fisico_data, schema=hub_stock_fisico_schema)
sat_articulo_detalles_df = spark.createDataFrame(data=sat_articulo_detalles_data, schema=sat_articulo_detalles_schema)
sat_tipo_articulo_detalles_df = spark.createDataFrame(data=sat_tipo_articulo_detalles_data, schema=sat_tipo_articulo_detalles_schema)
sat_stock_fisico_detalles_df = spark.createDataFrame(data=sat_stock_fisico_detalles_data, schema=sat_stock_fisico_detalles_schema)
lnk_articulo_tipo_articulo_df = spark.createDataFrame(data=lnk_articulo_tipo_articulo_data, schema=lnk_articulo_tipo_articulo_schema)
lnk_articulo_stock_fisico_df = spark.createDataFrame(data=lnk_articulo_stock_fisico_data, schema=lnk_articulo_stock_fisico_schema)
sat_lnk_articulo_stock_estado_df = spark.createDataFrame(data=sat_lnk_articulo_stock_estado_data, schema=sat_lnk_articulo_stock_estado_schema)

# Esto nos permite usar la sintaxis de SQL sobre nuestros DataFrames.
hub_articulo_df.createOrReplaceTempView("hub_articulo")
hub_tipo_articulo_df.createOrReplaceTempView("hub_tipo_articulo")
hub_stock_fisico_df.createOrReplaceTempView("hub_stock_fisico")
lnk_articulo_tipo_articulo_df.createOrReplaceTempView("lnk_articulo_tipo_articulo")
lnk_articulo_stock_fisico_df.createOrReplaceTempView("lnk_articulo_stock_fisico")
sat_articulo_detalles_df.createOrReplaceTempView("sat_articulo_detalles")
sat_tipo_articulo_detalles_df.createOrReplaceTempView("sat_tipo_articulo_detalles")
sat_stock_fisico_detalles_df.createOrReplaceTempView("sat_stock_fisico_detalles")
sat_lnk_articulo_stock_estado_df.createOrReplaceTempView("sat_lnk_articulo_stock_estado")

### Consigna 1: SparkQL que genere la foto del stock del día anterior.

In [26]:
###################################
# --- PASO 3: RESOLUCIÓN DE LAS CONSIGNAS ---
###################################

# =================================================================================
# Consigna 1: SparkQL que genere la foto del stock del día anterior.
# =================================================================================
print("\n--- Consigna 1: Datamart de Stock Diario ---")

query_stock_diario = """
    WITH StockConRanking AS (
        -- La lógica con ROW_NUMBER es idéntica a la de SQL estándar
        SELECT
            LK_ARTICULO_STOCK_PK,
            CantStock,
            Unidades,
            LOAD_DATE,
            ROW_NUMBER() OVER(PARTITION BY LK_ARTICULO_STOCK_PK ORDER BY LOAD_DATE DESC) as rn
        FROM
            sat_lnk_articulo_stock_estado
        WHERE
            -- Usamos las funciones de fecha de Spark
            date(LOAD_DATE) <= date_sub(current_date(), 1)
    )
    SELECT
        date_sub(current_date(), 1) as FechaReporte,
        ha.BK_IDARTICULO AS CodigoArticulo,
        sad.SectSecc AS DescripcionArticulo,
        hsf.BK_IDSTOCK AS CodigoUbicacion,
        ssfd.Descrip AS DescripcionUbicacion,
        scr.CantStock AS StockAlCierre,
        scr.Unidades
    FROM
        StockConRanking scr
    JOIN
        lnk_articulo_stock_fisico lasf ON scr.LK_ARTICULO_STOCK_PK = lasf.LK_ARTICULO_STOCK_PK
    JOIN
        hub_articulo ha ON lasf.HK_ARTICULO = ha.HK_ARTICULO
    JOIN
        sat_articulo_detalles sad ON ha.HK_ARTICULO = sad.HK_ARTICULO
    JOIN
        hub_stock_fisico hsf ON lasf.HK_STOCK_FISICO = hsf.HK_STOCK_FISICO
    JOIN
        sat_stock_fisico_detalles ssfd ON hsf.HK_STOCK_FISICO = ssfd.HK_STOCK_FISICO
    WHERE
        scr.rn = 1
"""

dm_stock_diario_df = spark.sql(query_stock_diario)
dm_stock_diario_df.show()


--- Consigna 1: Datamart de Stock Diario ---
+------------+--------------+-------------------+---------------+--------------------+-------------+--------+
|FechaReporte|CodigoArticulo|DescripcionArticulo|CodigoUbicacion|DescripcionUbicacion|StockAlCierre|Unidades|
+------------+--------------+-------------------+---------------+--------------------+-------------+--------+
|  2026-05-13|         75001|                  1|             10|    Depósito Central|          150|     150|
|  2026-05-13|         75001|                  1|             20|    Tienda Unicenter|           12|      12|
|  2026-05-13|         83045|                  2|             20|    Tienda Unicenter|           25|      25|
+------------+--------------+-------------------+---------------+--------------------+-------------+--------+



### Consigna 2: Un datamart aislado que contenga la info de los proveedores.

In [27]:
# =================================================================================
# Consigna 2: Un datamart aislado que contenga la info de los proveedores.
# =================================================================================
print("\n--- Consigna 2: Datamart de Proveedores ---")
print("No es posible generar este datamart.")
print("Razón: No existen datos de proveedores en las fuentes de origen cargadas en el Data Vault.")
print("Camino a seguir: 1. Identificar fuente de datos de proveedores. 2. Ingestar y modelar en el Data Vault (HUB_PROVEEDOR, etc.). 3. Construir el datamart a partir del Data Vault enriquecido.")



--- Consigna 2: Datamart de Proveedores ---
No es posible generar este datamart.
Razón: No existen datos de proveedores en las fuentes de origen cargadas en el Data Vault.
Camino a seguir: 1. Identificar fuente de datos de proveedores. 2. Ingestar y modelar en el Data Vault (HUB_PROVEEDOR, etc.). 3. Construir el datamart a partir del Data Vault enriquecido.


### Consigna 3: Un datamart que contenga la información de scanning.

In [28]:
# =================================================================================
# Consigna 3: Un datamart que contenga la información de scanning.
# =================================================================================
print("\n--- Consigna 3: Datamart de Scanning ---")

query_scanning = """
    SELECT
        salse.TS_ORIGEN AS FechaScanning,
        ha.BK_IDARTICULO AS CodigoArticulo,
        sad.SectSecc AS DescripcionArticulo,
        hsf.BK_IDSTOCK AS CodigoUbicacion,
        ssfd.Descrip AS DescripcionUbicacion,
        salse.Scanning,
        salse.Usuario,
        salse.Terminal
    FROM
        sat_lnk_articulo_stock_estado salse
    JOIN
        lnk_articulo_stock_fisico lasf ON salse.LK_ARTICULO_STOCK_PK = lasf.LK_ARTICULO_STOCK_PK
    JOIN
        hub_articulo ha ON lasf.HK_ARTICULO = ha.HK_ARTICULO
    JOIN
        sat_articulo_detalles sad ON ha.HK_ARTICULO = sad.HK_ARTICULO
    JOIN
        hub_stock_fisico hsf ON lasf.HK_STOCK_FISICO = hsf.HK_STOCK_FISICO
    JOIN
        sat_stock_fisico_detalles ssfd ON hsf.HK_STOCK_FISICO = ssfd.HK_STOCK_FISICO
    ORDER BY
        salse.TS_ORIGEN DESC
"""

dm_scanning_df = spark.sql(query_scanning)
dm_scanning_df.show()


# --- PASO 4: DETENER LA SESIÓN DE SPARK ---
spark.stop()


--- Consigna 3: Datamart de Scanning ---
+-------------+--------------+-------------------+---------------+--------------------+--------+-------+--------+
|FechaScanning|CodigoArticulo|DescripcionArticulo|CodigoUbicacion|DescripcionUbicacion|Scanning|Usuario|Terminal|
+-------------+--------------+-------------------+---------------+--------------------+--------+-------+--------+
|         NULL|         75001|                  1|             10|    Depósito Central|     123|   NULL|    NULL|
|         NULL|         75001|                  1|             10|    Depósito Central|     123|   NULL|    NULL|
|         NULL|         75001|                  1|             20|    Tienda Unicenter|     124|   NULL|    NULL|
|         NULL|         75001|                  1|             20|    Tienda Unicenter|     124|   NULL|    NULL|
|         NULL|         83045|                  2|             20|    Tienda Unicenter|     125|   NULL|    NULL|
|         NULL|         83045|                